In [ ]:
!pip install transformers
!pip install accelerate

In [ ]:
import pandas as pd

In [ ]:
import gdown
mental_health_data_link = 'https://drive.google.com/uc?export=view&id=11CGO0DV-E3EpeRo8cdfoE41TK63JO5fE'
gdown.download(mental_health_data_link, quiet=False)

Downloading...
From: https://drive.google.com/uc?export=view&id=11CGO0DV-E3EpeRo8cdfoE41TK63JO5fE
To: /content/mental_health_data.csv
100%|██████████| 4.64M/4.64M [00:00<00:00, 75.2MB/s]


'mental_health_data.csv'

In [ ]:
df = pd.read_csv("mental_health_data.csv", dtype = str)

In [ ]:
df.dropna(axis = 0, inplace = True)

In [ ]:
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3508 entries, 0 to 3511
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Context   3508 non-null   object
 1   Response  3508 non-null   object
dtypes: object(2)
memory usage: 82.2+ KB


In [ ]:
# Transform Context and Response so that all white space characters are converted to space
df["Context"] = df["Context"].apply(lambda x: " ".join(x.split()))
df["Response"] = df["Response"].apply(lambda x: " ".join(x.split()))

In [ ]:
patient_tag = "<|patient|>"
doctor_tag = "<|therapist|>"
df["patient_query_response"] = patient_tag + "\n" + df["Context"] + "\n" + doctor_tag + "\n" + df["Response"] + "<|endoftext|>\n"

In [ ]:
print(df["patient_query_response"].iloc[0])

<|patient|>
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?
<|therapist|>
If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Bad

In [ ]:
num_examples = df.shape[0]
num_training_examples = int(num_examples * 0.95)
num_test_examples = num_examples - num_training_examples

In [ ]:
training_set = df.patient_query_response.values[0: num_training_examples]
test_set = df.patient_query_response.values[num_training_examples:]

In [ ]:
print(len(training_set))
print(len(test_set))

3332
176


In [ ]:
with open('training_set.txt','w') as f:
  f.write(''.join(training_set))
with open('test_set.txt','w') as f:
  f.write(''.join(test_set))

In [ ]:
!head -20 training_set.txt

<|patient|>
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?
<|therapist|>
If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Bad

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
model = AutoModelForCausalLM.from_pretrained('distilgpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
input_text = 'I fell from my bicycle and I broke my'
enc_input = tokenizer.encode(input_text,return_tensors='pt', add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 70,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

0: I fell from my bicycle and I broke my ankle, and then I started running. I went to my doctor, and I told him that I was going to take a pill. So I was going to take it. So I took it and told him to stop taking it. I did it.



The other day I was walking

1: I fell from my bicycle and I broke my leg. He was on my way home to work.
"He was like, 'I don't know if it's okay to run but I don't know if it's alright to walk, so I don't know if it's okay to run, so I don't know if it's okay

2: I fell from my bicycle and I broke my leg. I got my first ride to my house on Thursday. I thought, what a crazy day it is, this was an awesome day for me. My family has been amazing.



I was driving a car from my house in the middle of the street in the middle of a busy

3: I fell from my bicycle and I broke my legs, fell out of the way, and had to do it to get back up," she said.




He said she tried to reach her husband when he refused to walk on the road and tried to pull her out of the way.




In [ ]:
input_text = test_set[0].split(doctor_tag)[0]  +  doctor_tag + "\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)

In [ ]:
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

0: <|patient|>
I have a friend that who I used to be in a relationship with. It was brief and turned into us being just good friends. I spent the weekend with him and it upset my boyfriend. Was i wrong?
<|therapist|>
I am really sorry for this. It's been very frustrating. I'm sorry. I've been through this with no regrets. I've had some bad decisions that have been a big part of my life. But it's not my fault that I did not get that. I'm sorry. I can't think of the worst.
<|therapist|>
I am sorry. I was just a little bit scared and scared. I was scared for

1: <|patient|>
I have a friend that who I used to be in a relationship with. It was brief and turned into us being just good friends. I spent the weekend with him and it upset my boyfriend. Was i wrong?
<|therapist|>
I am a very good person. It was a very good feeling to know that I was able to stay with him for about a week. He is a very good person and he is a very good person and I was very happy with him.
<|therapist|>
I feel ver

In [ ]:
#!curl https://webpages.scu.edu/ftp/msamorani/NLP/run_lm_finetuning.py > run_lm_finetuning.py
fixed_fine_tuning_code = 'https://drive.google.com/uc?export=view&id=1wJyT5k29A9tkpQV0OITdLHOpddFYjW4j'
gdown.download(fixed_fine_tuning_code, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?export=view&id=1wJyT5k29A9tkpQV0OITdLHOpddFYjW4j
From (redirected): https://drive.google.com/uc?export=view&id=1wJyT5k29A9tkpQV0OITdLHOpddFYjW4j&confirm=t&uuid=884ee585-91f9-4983-99eb-5cd9ea737491
To: /content/run_lm_finetuning.py
100%|██████████| 31.0k/31.0k [00:00<00:00, 88.9MB/s]


'run_lm_finetuning.py'

In [ ]:
!mkdir experiments

In [ ]:
epochs=105
file_with_training_set = 'training_set.txt'

In [ ]:
# this is bash code to fine-tune a language model. File from here: https://github.com/alontalmor/pytorch-transformers/blob/master/examples/run_lm_finetuning.py
text = f"for epoch in {epochs} \n"+\
"do \n"+\
"python run_lm_finetuning.py "+\
f"--output_dir=experiments/epoch_{epochs} "+\
"--model_type=gpt2 "+\
"--model_name_or_path=distilgpt2 "+\
f"--train_data_file={file_with_training_set} "+\
"--do_train "+\
"--overwrite_output_dir "+\
"--save_steps=5000 " +\
f"--num_train_epochs={epochs} \n" +\
"done"

In [ ]:
f = open('run_experiments.sh',mode='w')
f.write(text)
f.close()

In [ ]:
!bash run_experiments.sh

Streaming output truncated to the last 5000 lines.
Iteration:   8% 20/253 [00:01<00:13, 17.44it/s]
Iteration:   9% 22/253 [00:01<00:13, 17.44it/s]
Iteration:   9% 24/253 [00:01<00:13, 17.44it/s]
Iteration:  10% 26/253 [00:01<00:13, 17.44it/s]
Iteration:  11% 28/253 [00:01<00:12, 17.44it/s]
Iteration:  12% 30/253 [00:01<00:12, 17.44it/s]
Iteration:  13% 32/253 [00:01<00:12, 17.44it/s]
Iteration:  13% 34/253 [00:01<00:12, 17.44it/s]
Iteration:  14% 36/253 [00:02<00:12, 17.44it/s]
Iteration:  15% 38/253 [00:02<00:12, 17.44it/s]
Iteration:  16% 40/253 [00:02<00:12, 17.44it/s]
Iteration:  17% 42/253 [00:02<00:12, 17.44it/s]
Iteration:  17% 44/253 [00:02<00:11, 17.44it/s]
Iteration:  18% 46/253 [00:02<00:11, 17.44it/s]
Iteration:  19% 48/253 [00:02<00:11, 17.44it/s]
Iteration:  20% 50/253 [00:02<00:11, 17.44it/s]
Iteration:  21% 52/253 [00:02<00:11, 17.44it/s]
Iteration:  21% 54/253 [00:03<00:11, 17.44it/s]
Iteration:  22% 56/253 [00:03<00:11, 17.44it/s]
Iteration:  23% 58/253 [00:03<00:11, 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f'experiments/epoch_{epochs}')
model = AutoModelForCausalLM.from_pretrained(f'experiments/epoch_{epochs}')

model_path = f"mental_health_model_epoch_{epochs}"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
# mount it
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
# copy it as a new directory in the root of your google drive
import shutil
shutil.copytree(model_path,'/content/drive/MyDrive/'+ model_path)

In [ ]:
input_text = test_set[0].split(doctor_tag)[0]  +  doctor_tag + "\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')


In [ ]:
input_text = 'Today I fell from my bicycle and broke my '
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    max_length= 70,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

In [ ]:
input_text = "Patient:\nWhy is everything hard?\nDoctor:\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')
